# 02 · Etiquetado por reglas: intención, categoría y urgencia

**Objetivo:** asignar a cada tweet de **cliente** tres etiquetas usando diccionarios de palabras, y preparar una muestra para que tú la etiquetes a mano y poder medir qué tan buenas son las reglas.

**Importante para entender el flujo (son dos archivos distintos con propósitos distintos):**

| Archivo | Para qué sirve |
|---|---|
| `clientes_etiquetados.csv` | **Todos** los tweets con la etiqueta automática de las reglas. |
| `muestra_para_etiquetar.csv` | Una muestra pequeña **sin** etiquetas automáticas, para que tú pongas la etiqueta "verdadera" a mano. |
| `muestra_clave_reglas.csv` | La misma muestra **con** la etiqueta automática; se usa solo después, para comparar. |

**Cómo se eligen las categorías:** el código **no las descubre**: las definí yo como propuesta inicial en el diccionario `CATEGORIAS` (hardware, iOS, cuenta, apps, pagos, servicios, conectividad). Si un tweet no coincide con ninguna, queda en `otros`. En la sección 4 verás cómo comprobar si te faltan categorías.

## 1. Importaciones y rutas

In [1]:
import re
from collections import Counter
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report

IN = Path("../data/processed/apple_support_filtrado.parquet")
OUT = Path("../data/processed/clientes_etiquetados.parquet")
MUESTRA = Path("../data/processed/muestra_para_etiquetar.csv")       # En CSV para revisión manual
CLAVE = Path("../data/processed/muestra_clave_reglas.parquet")

## 2. Diccionarios de palabras
Cada diccionario es una lista de expresiones regulares. La función `rx` las une en una sola búsqueda que ignora mayúsculas y exige que la palabra **empiece** en un límite de palabra (`\b`), de modo que `crash` también detecta `crashes` pero `bug` no se activa dentro de otra palabra.

- **FALLA / SOLICITUD:** distinguen si el cliente reporta un problema o pide información.
- **CATEGORIAS:** el tema técnico. Es el diccionario que debes ampliar o editar.
- **CRITICA / MEDIA / AMPLIFICADORES:** definen el nivel de urgencia (ver abajo).

**Definición de urgencia (solo para fallas):**
- **Crítica:** pérdida total de servicio, acceso, datos, dinero o seguridad; o una falla media con apremio, duración o alcance (*still*, *for days*, *everyone*).
- **Media:** función degradada o parcial.
- **Baja:** solicitudes, dudas o fallas cosméticas.

TABLA DE DESCRIPCIÓN DE CATEGORÍAS TÉCNICAS:

| Categoría | Expresiones regulares representativas | Escenarios cubiertos |
|---|---|---|
| hardware_dispositivo | battery, screen, charge/charger, camera, speaker, button, touch id, face id, audio, mic | Fallas físicas del equipo: sobrecalentamiento, batería agotada/inflada, pantalla rota o no responsiva, problemas de puerto de carga, botones trabados o menciones directas a hardware. |
| ios_software_actualizacion | ios, update, upgrade, macos, high sierra, beta, firmware, keyboard, autocorrect, glitch, bug | Incidentes derivados de actualizaciones del sistema operativo, errores tras instalar parches, lentitud del sistema, bugs de interfaz (ej. bug del teclado / signo de interrogación) y bloqueos lógicos. |
| conectividad | wi-fi, bluetooth, cellular, signal, network, no service, sim, carrier, lte, phone call, calls | Pérdida de cobertura móvil, incapacidad de emitir o recibir llamadas telefónicas, fallas de emparejamiento Bluetooth, desconexiones de Wi-Fi y problemas con la tarjeta SIM o datos celulares. |
| cuenta_icloud | icloud, apple id, password, log in, sign in, locked, 2fa, verification, backup, storage | Cuentas bloqueadas por seguridad, olvido de credenciales, fallas de autenticación de dos factores, problemas de almacenamiento o sincronización de respaldos en la nube. |
| apps_appstore | app store, app/apps, download, testflight, safari, mail app, crash de apps de terceros | Aplicaciones que se cierran inesperadamente (crash), descargas bloqueadas en la tienda, fallas al abrir navegadores (Safari) o clientes de correo. |
| servicios_apple | apple music, playlist, imessage, facetime, siri, icloud mail, apple tv, podcasts, itunes | Problemas específicos con plataformas y servicios propios de Apple: canciones desaparecidas, iMessage no activa, llamadas de FaceTime caídas o fallas en Siri. |
| pagos_facturacion | charged, billing, refund, payment, subscription, invoice, apple pay, credit card, purchase | Cobros no reconocidos o duplicados, gestión de suscripciones periódicas, solicitudes de reembolso y errores en pasarelas de pago como Apple Pay. |
| otros | (Sin coincidencia en los términos anteriores) | Mensajes genéricos, quejas sin detalle técnico o vocabulario no cubierto en los diccionarios previos. |
"""

In [2]:
# ## 2. Diccionarios de palabras (Catálogo Ampliado)

def rx(palabras):
    return re.compile(r"\b(?:" + "|".join(palabras) + r")", re.IGNORECASE)

FALLA = rx([
    # Negaciones e incapacidad de uso (incluye el caso "not able to")
    r"not working", r"doesn'?t work", r"don'?t work", r"isn'?t working", r"aren'?t working",
    r"stopped working", r"quit working", r"never works", r"no longer (?:works?|working|tracks?|recogni[sz]es?)",
    r"not able to", r"unable to", r"can'?t", r"cannot", r"won'?t", r"will not",
    r"isn'?t (?:turning|charging|showing|allowing|letting|connecting|responding|syncing|opening|loading)",
    r"aren'?t (?:showing|updating|working|syncing|loading)",
    r"doesn'?t (?:want to|let me|show|recognize|charge|connect|turn on|appear|work)",
    r"refuses? to", r"stopping in the middle",
    # Fallos de sistema, bugs y glitches
    r"broken", r"crash", r"bug", r"buggy", r"error", r"glitch", r"issue", r"problem",
    r"fail", r"failure", r"malfunction", r"corrupt", r"messed up", r"acting (?:up|crazy)",
    # Bloqueos, lentitud y pantallas
    r"stuck", r"freez", r"frozen", r"unresponsive", r"hangs?\b", r"lag", r"lagging",
    r"slow", r"sluggish", r"crawl", r"black screen", r"blank (?:lock )?screen",
    r"spinning wheel", r"boot ?loop",
    # Energía, reinicios y caídas de servicio
    r"down\b", r"outage", r"drain", r"dying", r"exploded", r"overheat", r"cooking my cpu",
    r"reboot", r"restart", r"shuts? (?:down|off)", r"turning off",
    # Pérdida de datos o comportamiento repetitivo anómalo
    r"lost", r"missing", r"disappear", r"delet(?:ed|ing)", r"gone\b",
    r"keeps? (?:freezing|crashing|restarting|stopping|deleting|turning|shutting|appearing|dropping)",
])

SOLICITUD = rx([
    # Preguntas directas (con o sin signo de interrogación)
    r"how (?:do|can|to|does|much|about)", r"where (?:do|can|is|are|to)",
    r"what (?:is|does|are|do i|can i|should i)", r"when (?:will|is|does|can)",
    r"why (?:is|does|do|did|can'?t|won'?t)", r"which (?:one|is|model)",
    r"is there (?:a|any|an option)", r"is it (?:possible|normal|true|safe)",
    r"does (?:anyone|apple|it support)", r"do you (?:have|recycle|know|offer)",
    # Peticiones e intenciones sin signo de interrogación
    r"can i\b", r"could (?:you|i|u)\b", r"can (?:you|u|someone) (?:help|tell|explain|advise)",
    r"any (?:way|idea|tips?|suggestions?|explanations?|options?|recommendations?|updates?)",
    r"please (?:help|advise|tell|explain)", r"need (?:help|recommendations?|answers?|advice|to know)",
    r"(?:just )?wondering (?:how|if|why|what|whether)", r"looking for (?:a|an|help|advice)",
    r"anyone (?:know|else have)", r"question\b", r"ideas\b",
])

CATEGORIAS = {
    "ios_software_actualizacion": rx([
        r"ios", r"ios ?1[012]", r"update", r"updating", r"updated", r"upgrade", r"upgrading",
        r"downgrade", r"macos", r"high sierra", r"sierra", r"beta", r"install", r"os\b",
        r"os ?1[012]", r"firmware", r"keyboard", r"key board", r"autocorrect",
        r"glitch", r"bug", r"buggy", r"question (?:mark|box)", r"capital [\"“']?i[\"”']?",
        r"control center", r"lock ?screen", r"notifications?", r"settings", r"cursor",
        r"security patch", r"root user",
    ]),
    "conectividad": rx([
        r"wi-?fi", r"bluetooth", r"\bbt\b", r"cellular", r"signal", r"network", r"no service",
        r"data\b", r"sim\b", r"carrier", r"roaming", r"lte\b", r"4g\b", r"3g\b", r"5g\b",
        r"hotspot", r"phone calls?", r"(?:make|receive|incoming) calls?", r"\bcalls?\b",
        r"calling", r"gps\b", r"airdrop", r"car ?play",
    ]),
    "hardware_dispositivo": rx([
        r"battery", r"screen", r"charg(?:e|er|ing)", r"camera", r"speaker", r"button",
        r"cracked", r"port\b", r"cable", r"usb", r"adapter", r"headphone", r"earpods?",
        r"airpods?", r"powerbeats", r"touch ?id", r"face ?id", r"3d touch", r"taptic",
        r"home button", r"side button", r"volume", r"true ?tone", r"brightness",
        r"logic board", r"cpu\b", r"memory", r"delaminating", r"band\b", r"projector",
        r"external monitor", r"sound\b", r"audio", r"mic(?:rophone)?",
    ]),
    "cuenta_icloud": rx([
        r"icloud", r"apple ?id", r"password", r"log ?in", r"logged in", r"sign ?in",
        r"locked", r"verification", r"2fa", r"backup", r"backed up", r"storage",
        r"restore", r"contacts", r"account", r"sync(?:ing)?",
    ]),
    "apps_appstore": rx([
        r"app ?store", r"\bapps?\b", r"application", r"download", r"testflight",
        r"safari", r"mail app", r"final cut", r"fcpx", r"finder", r"dock\b",
        r"clock app", r"youtube", r"hulu", r"facebook", r"snapchat", r"twitter app",
        r"outlook", r"google maps",
    ]),
    "servicios_apple": rx([
        r"apple music", r"\bmusic\b", r"playlists?", r"shuffle", r"songs?",
        r"imessages?", r"facetime", r"siri", r"icloud mail", r"apple ?tv",
        r"podcasts?", r"\bmaps\b", r"itunes", r"news feed",
    ]),
    "pagos_facturacion": rx([
        r"charged", r"billing", r"refund", r"payment", r"subscription", r"invoice",
        r"apple ?pay", r"credit card", r"purchase", r"purchased", r"pre-?confirm",
        r"ordered", r"payment plan", r"pay (?:this|off)",
    ]),
}

# Menciones genéricas a equipos: solo asignan "hardware_dispositivo" si ninguna otra categoría específica coincidió
DISPOSITIVOS_GENERICOS = rx([
    r"iphone", r"ipad", r"macbook", r"\bmac\b", r"apple watch", r"\bwatch\b",
    r"ipod", r"homepod", r"apple desktop", r"\bphone\b", r"laptop", r"computer",
])

CRITICA = rx([
    r"outage", r"down for", r"\bis down\b", r"services? (?:is |are )?down",
    r"completely (?:dead|broken|unusable)", r"almost completely unusable",
    r"bricked", r"won'?t (?:turn on|boot|start|restart)", r"isn'?t turning on",
    r"boot ?loop", r"stuck (?:on|with) (?:apple )?logo", r"locked on opening screen",
    r"entirely unresponsive", r"exploded",
    r"data loss", r"lost (?:all|everything)", r"all my (?:photos|data|contacts|playlists)",
    r"half of my contacts disappeared", r"applications folder is all but empty",
    r"hacked", r"unauthori[sz]ed", r"stolen", r"fraud", r"locked out",
    r"root user with no password",
    r"can'?t (?:log ?in|sign ?in|access)", r"charged (?:twice|\d+)", r"disabled",
    r"(?:can'?t|cannot|won'?t|unable to|not able to) (?:make|receive|connect)(?: or (?:make|receive))? (?:any )?(?:phone )?calls?",
    r"911|emergency",
])

MEDIA = rx([
    r"not working", r"doesn'?t work", r"don'?t work", r"isn'?t working", r"stopped working",
    r"crash", r"freez", r"frozen", r"error", r"bug", r"buggy", r"glitch", r"stuck",
    r"slow", r"sluggish", r"lag", r"drain", r"dying", r"unresponsive",
    r"won'?t (?:update|install|download|connect|sync|activate|show|charge|lock|open|let me)",
    r"keeps", r"missing", r"disappear", r"fail", r"can'?t", r"cannot",
    r"unable to", r"not able to", r"no longer", r"broken",
])

AMPLIFICADORES = rx([
    r"urgent", r"asap", r"immediately", r"\bstill\b", r"for (?:days|hours|weeks|months)",
    r"since (?:yesterday|monday|this morning|last)", r"(?:\d+|two|three|ten) (?:days|weeks|months)",
    r"everyone", r"anyone else", r"all (?:users|of us)", r"again and again",
    r"over and over", r"twice a day", r"every time", r"constantly", r"regularly",
    r"work(?:ing)? from", r"business", r"third time", r"multiple updates",
])

## 3. Funciones de clasificación
Cada función recibe el texto de un tweet y devuelve una etiqueta:

- `intencion`: si hay palabra de falla → `falla`; si no, y hay patrón de pregunta o "?" → `solicitud`; si nada coincide → `otro`. (La falla tiene prioridad: "my phone won't charge, how do I fix it?" es una falla.)
- `categoria`: cuenta cuántas palabras de cada categoría aparecen y elige la que más tiene. Si ninguna coincide → `otros`. Si hay empate gana la primera del diccionario.
- `urgencia`: solicitudes → baja; falla con término crítico → crítica; falla con término medio → media (sube a crítica si hay amplificador); falla sin más → baja.

In [3]:
def intencion(t):
    if FALLA.search(t):
        return "falla"
    if SOLICITUD.search(t) or "?" in t:
        return "solicitud"
    return "otro"

def categoria(t):
    puntos = {c: len(r.findall(t)) for c, r in CATEGORIAS.items()}
    mejor = max(puntos, key=puntos.get)
    if puntos[mejor] > 0:
        return mejor
    if DISPOSITIVOS_GENERICOS.search(t):
        return "hardware_dispositivo"
    return "otros"

def urgencia(t, intent):
    if intent != "falla":
        return "baja"
    if CRITICA.search(t):
        return "critica"
    if MEDIA.search(t):
        return "critica" if AMPLIFICADORES.search(t) else "media"
    return "baja"

## 4. Aplicar las reglas a todos los tweets de clientes
Se carga el CSV del notebook 01, se filtran solo los clientes y se agregan tres columnas nuevas. El resultado completo se guarda en `clientes_etiquetados.csv`.

In [4]:
import pyarrow.parquet as pq

df = pq.read_table(IN).to_pandas()
cl = df[df["rol"] == "cliente"].copy()
t = cl["texto_limpio"].fillna("")

cl["intencion"] = t.apply(intencion)
cl["categoria"] = t.apply(categoria)
cl["urgencia"] = [urgencia(x, i) for x, i in zip(t, cl["intencion"])]
cl.to_parquet(OUT, index=False, engine="pyarrow", compression="snappy")
print(f"Dataset clasificado guardado en: {OUT.resolve()} ({len(cl):,} filas)")
cl[["texto_limpio", "intencion", "categoria", "urgencia"]].sample(10, random_state=1)

Dataset clasificado guardado en: C:\Users\julia\OneDrive\Escritorio\MCD\Tesis\proyecto_investigacion\data\processed\clientes_etiquetados.parquet (126,113 filas)


,texto_limpio,intencion,categoria,urgencia
226217,I was informed by @AppleSupport my problems af...,falla,ios_software_actualizacion,baja
22756,@AppleSupport Settings: Do Not Disturb is off....,otro,ios_software_actualizacion,baja
226573,@AppleSupport my iPhone 6S Plus keeps restarti...,falla,hardware_dispositivo,media
196248,Awful service from @AppleSupport in #CoventGar...,otro,hardware_dispositivo,baja
203950,".@AppleSupport, My iPhone SE ran perfectly aft...",falla,ios_software_actualizacion,critica
82334,@AppleSupport This would be great if anyone wo...,otro,conectividad,baja
225587,"@AppleSupport since 3 hours ago, it keeps rest...",falla,ios_software_actualizacion,critica
205072,@AppleSupport WhatsApp was the worst but apple...,falla,apps_appstore,media
82125,@AppleSupport is this legit or phishing? <URL>...,solicitud,otros,baja
203847,@AppleSupport I reset my password and the prob...,falla,cuenta_icloud,baja


## 5. Distribuciones (tablas para Resultados)
Muestran cuántos tweets cayeron en cada etiqueta. Si `otros` o `otro` es muy grande, las reglas no están cubriendo bien los datos.

In [5]:
print("\n--- Distribución de Intención ---")
print(cl["intencion"].value_counts(normalize=True).round(3))

print("\n--- Distribución de Urgencia en Fallas ---")
print(cl.loc[cl["intencion"] == "falla", "urgencia"].value_counts(normalize=True).round(3))

print("\n--- Tabla Cruzada Categoría vs. Urgencia ---")
print(pd.crosstab(cl["categoria"], cl["urgencia"]))


--- Distribución de Intención ---
intencion
falla        0.410
otro         0.392
solicitud    0.198
Name: proportion, dtype: float64

--- Distribución de Urgencia en Fallas ---
urgencia
media      0.565
baja       0.325
critica    0.110
Name: proportion, dtype: float64

--- Tabla Cruzada Categoría vs. Urgencia ---
urgencia                     baja  critica  media
categoria                                        
apps_appstore                3045      321   1600
conectividad                 3443      305   1087
cuenta_icloud                3364      418    933
hardware_dispositivo        17375     1102   4947
ios_software_actualizacion  27319     2906  16972
otros                       33132      454   2542
pagos_facturacion             882       32    183
servicios_apple              2658      161    932


## 6. ¿Las categorías están completas? Cómo comprobarlo
Las categorías son una hipótesis tuya, y los datos pueden contradecirla. Dos revisiones:

1. **Cobertura:** qué porcentaje de las fallas quedó en `otros`. Como referencia práctica, conviene que sea bajo (por ejemplo < 15-20 %); si es alto, faltan palabras o categorías.
2. **Palabras frecuentes en `otros`:** si aparece una palabra repetida (por ejemplo `homepod`, `carplay`), probablemente merece su categoría o entrar a una existente.

In [6]:
fallas = cl[cl["intencion"] == "falla"]
print(f"Fallas en 'otros': {(fallas['categoria'] == 'otros').mean():.1%}")

from sklearn.feature_extraction.text import CountVectorizer
otros = fallas.loc[fallas["categoria"] == "otros", "texto_limpio"]
vec = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=5)
conteos = vec.fit_transform(otros).sum(axis=0).A1
top = sorted(zip(vec.get_feature_names_out(), conteos), key=lambda x: -x[1])[:40]
pd.DataFrame(top, columns=["termino", "frecuencia"])

Fallas en 'otros': 13.7%


,termino,frecuencia
0,applesupport,5428
1,user,3786
2,fix,1529
3,problem,1520
4,issue,1382
5,url,946
6,user applesupport,667
7,just,561
8,applesupport user,553
9,apple,506


## 7. Ver ejemplos de cada etiqueta
Antes de etiquetar a mano, lee ejemplos por urgencia para decidir si las definiciones tienen sentido. Cambia el valor de `nivel` para ver los otros.

In [7]:
nivel = "critica"   # prueba: "media", "baja"
pd.set_option("display.max_colwidth", 200)
cl.loc[cl["urgencia"] == nivel, ["texto_limpio", "categoria"]].sample(15, random_state=3)

,texto_limpio,categoria
22640,@AppleSupport broken iPhone can't access iCloud. Refusing to stop charging for iCloud services,cuenta_icloud
137315,Why did I update my phone &amp; I'm still seeing question marks? Why is Siri not working? Why is my App Store not downloading things?@USER,ios_software_actualizacion
223331,@AppleSupport @USER Iphone 7 phone getting unresponsive every 5-10 mins after ios 11.1.2 update which promised to fix bugs. Please provide some resolution ASAP.,ios_software_actualizacion
226503,"@AppleSupport my iPhone 7 running iOS 11.1.2 just started crashing constantly, screen goes black then shows Apple busy circle symbol the auto locks my phone, brings me to the lock screen and after...",ios_software_actualizacion
157059,Only @USER would send me to a verified Apple tech only for the tech to tell me he can't do shit! My warranty expires in 5 days. Apple explain yourself,otros
169835,"@AppleSupport It was due to a bug in MacOS and/or iMovie app, since last weeks updates it finally works again. Pfeew",ios_software_actualizacion
229591,"Still convinced new updates are killing my previously great iPhone battery... @AppleSupport Drains super fast &amp; charged from 66% to 84% in 1.5hrs...on low battery mode, no apps running. <URL>",hardware_dispositivo
28837,"@AppleSupport Hi, I'm trying to get help with my phone as I can't make or receive calls but the only option I have is to call, help!",conectividad
9982,@USER @AppleSupport the new OS is TERRIBLE. It freezes and locks my phone constantly. So many issues it's shocking it was released,ios_software_actualizacion
63855,"@AppleSupport I've got an iPhone 8 and in the last two days the battery is draining like crazy. Went to bed with 100%, woke up with 35%",hardware_dispositivo


## 8. Muestra para etiquetado manual (a ciegas)
Se toma una muestra estratificada (hasta 40 tweets por combinación intención × urgencia, para que haya ejemplos de los casos raros como los críticos).

Se generan **dos** archivos:
- `muestra_para_etiquetar.csv`: solo `tweet_id`, texto y columnas vacías `*_manual`. **No incluye la etiqueta de las reglas**, para que no te influya al etiquetar.
- `muestra_clave_reglas.csv`: los mismos tweets con la etiqueta automática, que se usa en el paso 9.

En `comentario_error` anota por qué un tweet es difícil o ambiguo; esos comentarios alimentan tu subsección de errores y aciertos.

In [14]:
muestra = (cl.sample(frac=1, random_state=42)
             .groupby(["intencion", "urgencia"])
             .head(40))

para_etiquetar = muestra[["tweet_id", "texto_limpio"]].copy()
for c in ["intencion_manual", "categoria_manual", "urgencia_manual", "comentario_error"]:
    para_etiquetar[c] = ""

# Se guarda en CSV para que puedas rellenarlo en cualquier hoja de cálculo
para_etiquetar.to_csv(MUESTRA, index=False)
# La clave automática se guarda en Parquet (o CSV según tu preferencia)
muestra[["tweet_id", "intencion", "categoria", "urgencia"]].to_parquet(CLAVE, index=False)
print(f"\nMuestra generada para etiquetado ciego: {len(muestra)} tweets en {MUESTRA.name}")


Muestra generada para etiquetado ciego: 200 tweets en muestra_para_etiquetar.csv


## 9. Evaluación contra tu etiquetado manual (hacer después de etiquetar)
Cuando termines de llenar `muestra_para_etiquetar.csv` (guárdalo como CSV), esta celda compara tus etiquetas con las de las reglas y calcula precisión, sensibilidad y **F1** por clase. Es la métrica de tu hipótesis (≥ 80 %).

Valores válidos al etiquetar: intención `falla`/`solicitud`/`otro`; urgencia `baja`/`media`/`critica`; categoría, los nombres del diccionario u `otros`.

In [ ]:
from sklearn.metrics import classification_report

manual = pd.read_csv(MUESTRA, dtype={"tweet_id": str})
clave = pq.read_table(CLAVE, dtype={"tweet_id": str})
m = manual.merge(clave, on="tweet_id")
m = m[m["intencion_manual"].notna() & (m["intencion_manual"] != "")]

if m.empty:
    print("Aún no hay etiquetas manuales.")
else:
    for dim in ["intencion", "categoria", "urgencia"]:
        print(f"\n=== {dim} ===")
        print(classification_report(m[f"{dim}_manual"], m[dim], zero_division=0))

TypeError: Can only merge Series or DataFrame objects, a <class 'pyarrow.lib.Table'> was passed